In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys

import os
from pathlib import Path

!pip install -q kaggle
import kagglehub

goemotions_path = kagglehub.dataset_download('debarshichanda/goemotions')

PROJECT_ROOT = Path(os.path.abspath('')).parent.parent
sys.path.append(str(PROJECT_ROOT))

from core.cleaning import clean_text, SLANG_DICT
from core.data_loaders import load_goemotions
from core.utils import print_dataset_stats
from sklearn.model_selection import train_test_split
from tqdm import tqdm  
import pandas as pd
import numpy as np
import psutil

from sklearn.model_selection import train_test_split

import pandas as pd
import polars as pl
from polars.exceptions import ShapeError
import numpy as np
import psutil
import re

memory = psutil.virtual_memory()
print(f"Total RAM: {memory.total / 1e9:.2f} GB")
print(f"Available RAM: {memory.available / 1e9:.2f} GB")
print(f"RAM Usage: {memory.percent}%")



os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DATA_PATH = '/root/.cache/kagglehub/datasets/debarshichanda/goemotions/versions/6/data/full_dataset'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'goemotions' dataset.
Total RAM: 13.61 GB
Available RAM: 11.95 GB
RAM Usage: 12.1%


## Loading Goemotions Dataset

In [ ]:
goemotions_df, emotion_cols = load_goemotions(DATA_PATH)

print(f"\nDataset shape: {goemotions_df.shape}")
print(f"Number of emotion classes: {len(emotion_cols)}")
print(f"Sample emotions: {emotion_cols[:10]}")

print("\nBasic statistics:")
print(f"- Total samples: {len(goemotions_df)}")
print(f"- Average emotions per sample: {goemotions_df[emotion_cols].sum(axis=1).mean():.2f}")
print(f"- Max emotions in a sample: {goemotions_df[emotion_cols].sum(axis=1).max()}")
print(f"- Min emotions in a sample: {goemotions_df[emotion_cols].sum(axis=1).min()}")



print("\n" + "="*60)
print("FIXING DUPLICATE TEXTS - AGGREGATING EMOTIONS")
print("="*60)

print(f"Original shape: {goemotions_df.shape}")
print(f"Original rows: {len(goemotions_df)}")
print(f"Unique texts: {goemotions_df['text'].nunique()}")


emotion_cols = [col for col in goemotions_df.columns if col not in ['text', 'example_very_unclear']]


goemotions_aggregated = goemotions_df.groupby('text').agg({
    **{col: 'max' for col in emotion_cols},
 'example_very_unclear': 'max' if 'example_very_unclear' in goemotions_df.columns else None
}).reset_index()


goemotions_aggregated = goemotions_aggregated.dropna(axis=1, how='all')

print(f"\n Aggregated shape: {goemotions_aggregated.shape}")
print(f" Aggregated rows: {len(goemotions_aggregated)}")
print(f" Removed {len(goemotions_df) - len(goemotions_aggregated)} duplicate rows")


goemotions_df = goemotions_aggregated


emotion_cols = [col for col in emotion_cols if col in goemotions_df.columns]
print(f"\n Now have {len(goemotions_df)} unique texts with {len(emotion_cols)} emotion columns")


Dataset shape: (211225, 30)
Number of emotion classes: 29
Sample emotions: ['example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire']

Basic statistics:
- Total samples: 211225
- Average emotions per sample: 1.20
- Max emotions in a sample: 12
- Min emotions in a sample: 1

FIXING DUPLICATE TEXTS - AGGREGATING EMOTIONS
Original shape: (211225, 30)
Original rows: 211225
Unique texts: 57731

Aggregated shape: (57731, 30)
Aggregated rows: 57731
Removed 153494 duplicate rows

Now have 57731 unique texts with 28 emotion columns


In [3]:

print("\n" + "="*60)
print("EMOTION DISTRIBUTION (BEFORE CLEANING)")
print("="*60)

emotion_sums_before = goemotions_df[emotion_cols].sum().sort_values(ascending=False)

print("\nTop 10 most frequent emotions:")
for emotion, count in emotion_sums_before.head(10).items():
    print(f" {emotion}: {count} ({count/len(goemotions_df)*100:.2f}%)")

print("\nBottom 10 least frequent emotions:")
for emotion, count in emotion_sums_before.tail(10).items():
    print(f" {emotion}: {count} ({count/len(goemotions_df)*100:.2f}%)")



EMOTION DISTRIBUTION (BEFORE CLEANING)

Top 10 most frequent emotions:
  neutral: 31445 (54.47%)
  approval: 13235 (22.93%)
  annoyance: 10024 (17.36%)
  admiration: 9912 (17.17%)
  disapproval: 8399 (14.55%)
  realization: 7248 (12.55%)
  disappointment: 6656 (11.53%)
  curiosity: 6203 (10.74%)
  optimism: 6199 (10.74%)
  joy: 5688 (9.85%)

Bottom 10 least frequent emotions:
  disgust: 4053 (7.02%)
  surprise: 3823 (6.62%)
  desire: 2838 (4.92%)
  fear: 2136 (3.70%)
  embarrassment: 2003 (3.47%)
  remorse: 1663 (2.88%)
  nervousness: 1556 (2.70%)
  pride: 1127 (1.95%)
  relief: 1085 (1.88%)
  grief: 558 (0.97%)


In [ ]:

print(f"\nCleaning {len(goemotions_df)} texts...")
tqdm.pandas(desc="Cleaning texts")
goemotions_df['text_clean'] = goemotions_df['text'].progress_apply(
    lambda x: clean_text(x, aggressive=False, slang_dict=SLANG_DICT)
)


original_len = len(goemotions_df)
goemotions_clean = goemotions_df[goemotions_df['text_clean'].str.len() > 10].reset_index(drop=True)

print(f"\n Processed {len(goemotions_clean)} rows")
print(f" Removed {original_len - len(goemotions_clean)} rows (empty after cleaning)")



before_duplicates = len(goemotions_clean)
goemotions_clean = goemotions_clean.drop_duplicates(subset=['text_clean'])
print(f" Removed {before_duplicates - len(goemotions_clean)} duplicate rows")


print(f"\nNull values before:")
print(goemotions_clean[['text_clean'] + emotion_cols].isnull().sum().sum())

goemotions_clean = goemotions_clean.dropna(subset=['text_clean'] + emotion_cols)
print(f" Removed null rows")


before_no_emotions = len(goemotions_clean)
emotion_sum = goemotions_clean[emotion_cols].sum(axis=1)
goemotions_clean = goemotions_clean[emotion_sum > 0]
print(f" Removed {before_no_emotions - len(goemotions_clean)} samples with no emotions")


if 'example_very_unclear' in goemotions_clean.columns:
    before_unclear = len(goemotions_clean)
    goemotions_clean = goemotions_clean[goemotions_clean['example_very_unclear'] == False]
    print(f" Removed {before_unclear - len(goemotions_clean)} unclear samples (example_very_unclear = True)")


print("\n" + "="*50)
print("FINAL DATA QUALITY REPORT")
print("="*50)
print(f"Total samples after all cleaning: {len(goemotions_clean)}")
print(f"Duplicates: {goemotions_clean.duplicated(subset=['text_clean']).sum()}")
print(f"Nulls: {goemotions_clean[emotion_cols].isnull().sum().sum()}")
print(f"Average emotions per sample: {goemotions_clean[emotion_cols].sum(axis=1).mean():.2f}")
print(f"Max emotions in a sample: {goemotions_clean[emotion_cols].sum(axis=1).max()}")


goemotions_clean['text'] = goemotions_clean['text_clean']
goemotions_clean = goemotions_clean.drop(columns=['text_clean'])


Cleaning 57731 texts...


Cleaning texts:   0%|          | 0/57731 [00:00<?, ?it/s]

Loading spelling correction model on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Cleaning texts:   0%|          | 255/57731 [00:07<18:49, 50.86it/s] 

Failed to load spelling correction model: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"
Spelling correction disabled


Cleaning texts: 100%|██████████| 57731/57731 [00:20<00:00, 2783.68it/s]



Processed 57052 rows
Removed 679 rows (empty after cleaning)
Removed 95 duplicate rows

Null values before:
0
Removed null rows
Removed 2 samples with no emotions
Removed 2587 unclear samples (example_very_unclear = True)

FINAL DATA QUALITY REPORT
Total samples after all cleaning: 54368
Duplicates: 0
Nulls: 0
Average emotions per sample: 2.88
Max emotions in a sample: 13


In [ ]:


emotion_sums_after = goemotions_clean[emotion_cols].sum().sort_values(ascending=False)

print("\nTop 10 most frequent emotions (after cleaning):")
for emotion, count in emotion_sums_after.head(10).items():
    print(f" {emotion}: {count} ({count/len(goemotions_clean)*100:.2f}%)")

print("\nBottom 10 least frequent emotions (after cleaning):")
for emotion, count in emotion_sums_after.tail(10).items():
    print(f" {emotion}: {count} ({count/len(goemotions_clean)*100:.2f}%)")



Top 10 most frequent emotions (after cleaning):
  neutral: 29275 (53.85%)
  approval: 12682 (23.33%)
  admiration: 9544 (17.55%)
  annoyance: 9462 (17.40%)
  disapproval: 7974 (14.67%)
  realization: 6986 (12.85%)
  disappointment: 6369 (11.71%)
  optimism: 5991 (11.02%)
  curiosity: 5908 (10.87%)
  joy: 5393 (9.92%)

Bottom 10 least frequent emotions (after cleaning):
  disgust: 3879 (7.13%)
  surprise: 3550 (6.53%)
  desire: 2744 (5.05%)
  fear: 2066 (3.80%)
  embarrassment: 1926 (3.54%)
  remorse: 1588 (2.92%)
  nervousness: 1506 (2.77%)
  pride: 1092 (2.01%)
  relief: 1047 (1.93%)
  grief: 526 (0.97%)


In [ ]:

common_emotions = set(emotion_sums_before.index) & set(emotion_sums_after.index)
correlation = None
if common_emotions and len(common_emotions) > 1:
    before_values = [emotion_sums_before[e] for e in common_emotions]
    after_values = [emotion_sums_after[e] for e in common_emotions]

    correlation = np.corrcoef(before_values, after_values)[0, 1]
    print(f"\n Distribution correlation: {correlation:.3f}")
    if correlation > 0.99:
        print(" Emotion distributions are almost identical!")
    elif correlation > 0.95:
        print(" Emotion distributions are well preserved")
    else:
        print(" Emotion distributions show some change")


emotions_lost = set(emotion_sums_before[emotion_sums_before > 0].index) - set(emotion_sums_after[emotion_sums_after > 0].index)
if emotions_lost:
    print(f"\n Emotions completely lost: {emotions_lost}")
else:
    print(f"\n All emotions present in cleaned data")



Distribution correlation: 1.000
Emotion distributions are almost identical!

All emotions present in cleaned data


In [ ]:

train_df, temp_df = train_test_split(
    goemotions_clean,
    test_size=0.2,
    random_state=42,
    stratify=None  
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print(f"\n Final dataset sizes:")
print(f" Train: {len(train_df)} samples ({len(train_df)/len(goemotions_clean)*100:.1f}%)")
print(f" Validation: {len(val_df)} samples ({len(val_df)/len(goemotions_clean)*100:.1f}%)")
print(f" Test: {len(test_df)} samples ({len(test_df)/len(goemotions_clean)*100:.1f}%)")


print("\n Emotion distribution across splits (top 5):")
for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name}:")
    emotion_counts = split_df[emotion_cols].sum().sort_values(ascending=False)
    for emotion, count in emotion_counts.head(5).items():
        pct = count/len(split_df)*100
        print(f" {emotion}: {count} ({pct:.2f}%)")



Final dataset sizes:
  Train: 43494 samples (80.0%)
  Validation: 5437 samples (10.0%)
  Test: 5437 samples (10.0%)

Emotion distribution across splits (top 5):

Train:
  neutral: 23378 (53.75%)
  approval: 10174 (23.39%)
  admiration: 7652 (17.59%)
  annoyance: 7538 (17.33%)
  disapproval: 6395 (14.70%)

Validation:
  neutral: 2976 (54.74%)
  approval: 1224 (22.51%)
  admiration: 960 (17.66%)
  annoyance: 953 (17.53%)
  disapproval: 798 (14.68%)

Test:
  neutral: 2921 (53.72%)
  approval: 1284 (23.62%)
  annoyance: 971 (17.86%)
  admiration: 932 (17.14%)
  disapproval: 781 (14.36%)


In [ ]:


sample_size = min(5, len(goemotions_clean))
if sample_size > 0:
    sample_indices = np.random.choice(len(goemotions_clean), size=sample_size, replace=False)

    for idx in sample_indices:
        print("\n" + "-"*80)
        print(f"EXAMPLE (index in cleaned data: {idx})")
        print("-"*80)

        cleaned_text = goemotions_clean.iloc[idx]['text']

        
        emotions = []
        for col in emotion_cols:
            if col in goemotions_clean.columns and goemotions_clean.iloc[idx][col] == 1:
                emotions.append(col)

        print(f"\n CLEANED TEXT:")
        print(cleaned_text[:300] + "..." if len(cleaned_text) > 300 else cleaned_text)

        print(f"\n EMOTIONS ({len(emotions)}): {', '.join(emotions) if emotions else 'No emotions'}")



--------------------------------------------------------------------------------
EXAMPLE (index in cleaned data: 8480)
--------------------------------------------------------------------------------

CLEANED TEXT:
for be just being a handsome guy works wonders.

EMOTIONS (2): admiration, pride

--------------------------------------------------------------------------------
EXAMPLE (index in cleaned data: 46250)
--------------------------------------------------------------------------------

CLEANED TEXT:
yah, i'm a bad girl. in fact i'm bad at everything.

EMOTIONS (5): admiration, annoyance, approval, disgust, realization

--------------------------------------------------------------------------------
EXAMPLE (index in cleaned data: 33502)
--------------------------------------------------------------------------------

CLEANED TEXT:
small btc withdrawals are slowly happening. really curious to know if any large btc withdrawals have been successful.

EMOTIONS (2): curiosity, opti

In [ ]:

print("\n" + "="*60)
print("SAVING TO GOOGLE DRIVE")
print("="*60)


train_file = f'{DATA_PATH}/goemotions_train.csv'
train_df.to_csv(train_file, index=False)
print(f" Saved training set to {train_file}")

val_file = f'{DATA_PATH}/goemotions_val.csv'
val_df.to_csv(val_file, index=False)
print(f" Saved validation set to {val_file}")

test_file = f'{DATA_PATH}/goemotions_test.csv'
test_df.to_csv(test_file, index=False)
print(f" Saved test set to {test_file}")


emotion_file = f'{DATA_PATH}/goemotions_emotions.txt'
with open(emotion_file, 'w') as f:
    f.write('\n'.join(emotion_cols))
print(f" Saved emotion list to {emotion_file}")

print("\n" + "="*60)
print(" GOEMOTIONS DATA PREPARATION COMPLETE!")
print("="*60)
print(f"\nAll files saved to: {DATA_PATH}")